In [1]:
import pandas as pd

In [3]:
path_to_chembl = 'final_high_conf.tsv.gz'
topk = 10

In [4]:
chembl = pd.read_csv(path_to_chembl, delimiter='\t')
# basic filtering
chembl = chembl.loc[chembl.organism == 'Homo sapiens']
chembl = chembl.dropna(subset='pPot_mean')
# get accession of interest (largest ones)
accs_of_interest = chembl.accession.value_counts()[:topk].index.to_list()

In [5]:
accs_of_interest

['P00918',
 'P00915',
 'Q16790',
 'O43570',
 'P22303',
 'P35968',
 'Q13547',
 'P00742',
 'P06276',
 'P00533']

In [ ]:
# P00918: CAH2
# P00915: CAH1
# Q16790: CAH9
# O43570: CAH12
# P22303: ACES
# P35968: VGFR2
# Q13547: HDAC1
# P00742: FA10
# P06276: CHLE
# P00533: EGFR

Now we choose a subset of the above accession keys in order to get a more diverse set of targets

# Regression Datasets

In [11]:
# accession and standard_type to use
# hand picked based on the numer of compounds as well as diverse target space
accs_regression = [('P00918', 'Kd'),
                   ('P22303', 'IC50'),
                   ('P35968', 'IC50'),
                   ('Q13547', 'IC50'),
                   ('P00742', 'Kd'),
                   ('P06276', 'IC50'),
                   ('P00533', 'IC50'),
                   ]

cols_to_keep = ['nonstereo_aromatic_smiles', 'accession', 'standard_type', 'pPot_mean', 'chembl_cid', 'chembl_tid']

In [12]:
for acc, st in accs_regression:
    df = chembl.loc[(chembl.accession == acc) & (chembl.standard_type == st)]
    df[cols_to_keep].to_csv(f'regression_new/{acc}_{st}.csv', index=False)

# Classification Datasets

Use the same accession keys here for classification. As negative samples we will simply pick random compounds from ChEMBL which aren't active against the accession key of course

In [14]:
cols_to_keep = ['nonstereo_aromatic_smiles', 'accession', 'standard_type', 'chembl_cid', 'chembl_tid', 'label']

In [16]:
for acc, _ in accs_regression:
    positive_samples = chembl.loc[chembl.accession == acc].drop_duplicates(subset='nonstereo_aromatic_smiles')
    pool_of_negative_samples = chembl.loc[~chembl.nonstereo_aromatic_smiles.isin(positive_samples.nonstereo_aromatic_smiles)].drop_duplicates(subset='nonstereo_aromatic_smiles')

    negative_samples = pool_of_negative_samples.sample(n=positive_samples.shape[0],
                                                       replace=False,
                                                       random_state=42,
                                                       )

    positive_samples['label'] = 1
    negative_samples['label'] = 0

    df_comb = pd.concat((positive_samples, negative_samples))
    df_comb[cols_to_keep].to_csv(f'classification/{acc}.csv', index=False)